In [ ]:
# Colab runtime / NVIDIA GPU inventory (display only)
# This cell reports Colab hardware/runtime details only. It does not change training config.
import sys
import platform
import shutil
import subprocess
from pathlib import Path

def _run_cmd(cmd, timeout=20):
    exe = cmd[0]
    if shutil.which(exe) is None:
        return f"{exe} not found"
    try:
        result = subprocess.run(cmd, text=True, capture_output=True, timeout=timeout, check=False)
        output = (result.stdout or result.stderr or "").strip()
        return output if output else f"{exe} returned no output"
    except Exception as exc:
        return f"{exe} failed: {type(exc).__name__}: {exc}"


print(f"python: {sys.version.split()[0]}")
print(f"platform: {platform.platform()}")
print(f"cwd: {Path.cwd()}")

# Torch is inspected after installs, not here, to avoid loading it before pip changes.

print("\n--- nvidia-smi gpu summary ---")
print(_run_cmd([
    "nvidia-smi",
    "--query-gpu=index,name,memory.total,memory.used,driver_version,pstate,temperature.gpu,power.draw,power.limit",
    "--format=csv,noheader",
]))

print("\n--- nvidia-smi full ---")
print(_run_cmd(["nvidia-smi"]))

print("\n--- disks ---")
print(_run_cmd(["df", "-h"]))

print("\n--- memory ---")
print(_run_cmd(["free", "-h"]))


# Colab Short Trace Train And Submit

Train a Nemotron LoRA adapter with compact procedural traces for parseable cipher rows, then package one run bundle.

All settings are in the next cell.


## 1A. Experiment Config


In [ ]:
# Change this cell for each experiment.
EXPERIMENT_NAME = "S6_short_trace_boxed_r8_attention_drive"
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

PROMPT_EXPERIMENT = "short_trace_boxed"
SYSTEM_PROMPTS = {
    "direct_raw_0_62": "Solve the puzzle and output only the final answer value. Do not explain. Do not write prefixes like 'answer:' or 'the answer is'.",
    "boxed_final": "Solve the puzzle carefully. Output only the final answer inside \\boxed{} with no trailing text.",
    "private_reasoning_boxed": "Solve the puzzle carefully. Reason internally if needed, but write only the final answer inside \\boxed{} with no trailing text.",
    "short_trace_boxed": "Solve the puzzle. If useful, write a compact procedural trace using only the needed steps. End with exactly one final answer inside \\boxed{}.",
}
SYSTEM_PROMPT = SYSTEM_PROMPTS[PROMPT_EXPERIMENT]
TRAIN_TARGET_FORMAT = "short_trace_boxed"
TRACE_TARGET_MODE = "cipher_short_trace_else_boxed"

MAX_SEQ_LENGTH = 768
MAX_NEW_TOKENS = 256
TRAIN_ROWS = None
EVAL_ROWS = 256
PROBE_ROWS = 5
LOG_EVAL_POINTS = 8
SAVE_POINTS = 4

LORA_R = 8
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["in_proj", "out_proj", "q_proj", "k_proj", "v_proj", "o_proj"]
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 6
EPOCHS = 1
LEARNING_RATE = 3e-4
PRECISION_MODE = "bf16"

RUN_GENERATED_EVAL = True
GENERATED_EVAL_ROWS_ON_SAVE = EVAL_ROWS  # Use 0 to disable; EVAL_ROWS scores the full fixed eval split at saved checkpoints.

RESUME_FROM_CHECKPOINT = False
RESUME_CHECKPOINT_STEP = None  # Example: 192 when RESUME_FROM_CHECKPOINT=True.


## 1B. Fixed Paths And Constants


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/nemotron_challenge")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Colab_Notebooks/Kaggle challenges/nemotron_challenge/artefacts")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
ZIP_PATH = Path("/content/nvidia-nemotron-model-reasoning-challenge.zip")
OUTPUT_DIR = DRIVE_PROJECT_ROOT / "outputs" / EXPERIMENT_NAME
ADAPTER_DIR = OUTPUT_DIR / "adapter"
SANITY_CSV_PATH = OUTPUT_DIR / "sanity_test_predictions.csv"
SANITY_RAW_CSV_PATH = OUTPUT_DIR / "sanity_test_predictions_raw.csv"
RUN_BUNDLE_ZIP_PATH = DRIVE_PROJECT_ROOT / f"{EXPERIMENT_NAME}_run_bundle.zip"
RUN_CONFIG_PATH = OUTPUT_DIR / "run_config.json"
PROBE_SET_PATH = OUTPUT_DIR / "probe_questions.csv"
PROBE_EVOLUTION_CSV_PATH = OUTPUT_DIR / "probe_evolution.csv"
TRAINER_LOG_CSV_PATH = OUTPUT_DIR / "trainer_log_history.csv"
GENERATED_EVAL_CSV_PATH = OUTPUT_DIR / "generated_eval_predictions.csv"
GENERATED_EVAL_SUMMARY_PATH = OUTPUT_DIR / "generated_eval_summary.csv"
CHECKPOINT_EVAL_DIR = OUTPUT_DIR / "checkpoint_eval"
CHECKPOINT_EVAL_SUMMARY_PATH = CHECKPOINT_EVAL_DIR / "checkpoint_generated_eval_summary.csv"

TRACE_SAMPLE_PATH = OUTPUT_DIR / "trace_training_samples.csv"

if LORA_R > 32:
    raise ValueError("Kaggle submissions require LoRA rank <= 32")


## 2. Install and import

In [ ]:
!pip install packaging ninja
!pip install -q mamba-ssm --no-build-isolation
!pip uninstall -y -q causal-conv1d
!pip install --upgrade datasets
!pip install --upgrade transformers
!pip install --upgrade torchao

!pip install --upgrade peft
!pip install --upgrade trl
!pip install bitsandbytes
!pip install accelerate

!pip install tensorboard


In [ ]:
import json
import re
import time
import types
import zipfile

import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from google.colab import drive, files, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive output root:", DRIVE_PROJECT_ROOT)

HF_TOKEN = userdata.get("hftoken")


## 3. Load data

In [ ]:
if not (RAW_DIR / "train.csv").exists() or not (RAW_DIR / "test.csv").exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(RAW_DIR)

train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

assert list(train.columns) == ["id", "prompt", "answer"]
assert list(test.columns) == ["id", "prompt"]
print("train:", train.shape, "test:", test.shape)
display(test)


## 4. Prepare data

In [ ]:
def infer_family(prompt: str) -> str:
    text = str(prompt).lower()
    if "bit manipulation" in text or "8-bit binary" in text:
        return "bit_manipulation"
    if "secret encryption" in text or "decrypt" in text:
        return "cipher"
    if "numeral system" in text:
        return "numeral"
    if "unit" in text and "convert" in text:
        return "unit_conversion"
    if "gravitational constant" in text or "falling distance" in text:
        return "gravity"
    if "transformation rules is applied to equations" in text or "determine the result for" in text:
        return "equation"
    return "unknown"


def normalize_answer(value) -> str:
    text = "" if value is None else str(value).strip()
    text = re.sub(r"^`+|`+$", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip(".")


def extract_final_answer(text: str) -> str:
    """Extract a final answer using Kaggle's boxed-answer priority.

    The official metric first looks for answers in ``\\boxed{...}``. A
    literal ``}`` can be part of some answers, so this uses the last closing
    brace before the next boxed answer/end of text instead of stopping at the
    first ``}``.
    """
    text = "" if text is None else str(text)
    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    if boxed_starts:
        matches = []
        for idx, match in enumerate(boxed_starts):
            start = match.end()
            end = boxed_starts[idx + 1].start() if idx + 1 < len(boxed_starts) else len(text)
            segment = text[start:end]
            last_brace = segment.rfind("}")
            matches.append(segment[:last_brace] if last_brace != -1 else segment)
        non_empty = [match.strip() for match in matches if match.strip()]
        return normalize_answer(non_empty[-1] if non_empty else matches[-1])

    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
        r"final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return normalize_answer(matches[-1])

    number_matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if number_matches:
        return normalize_answer(number_matches[-1])

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        line = re.sub(r"^(final answer|answer|assistant)\s*[:\-]\s*", "", line, flags=re.I).strip()
        if line:
            return normalize_answer(line.strip("`").strip())
    return normalize_answer(text)


def build_cipher_short_trace(prompt: str, answer_text: str) -> str | None:
    """Build a compact, supervised trace for parseable substitution-cipher rows.

    The trace is intentionally short: it records only the target-word mappings
    needed to reach the answer, then leaves the final answer to the boxed line.
    This teaches a procedure without training the model to write long essays.
    """
    prompt_text = "" if prompt is None else str(prompt)
    if "secret encryption rules" not in prompt_text.lower():
        return None

    pairs = []
    for line in prompt_text.splitlines():
        if "->" not in line:
            continue
        left, right = line.split("->", 1)
        left = re.sub(r"^\s*[-*?]?\s*", "", left).strip()
        right = right.strip()
        if left and right:
            pairs.append((left, right))
    if not pairs:
        return None

    char_map = {}
    word_map = {}
    conflicts = set()
    for cipher_text, plain_text in pairs:
        cipher_words = cipher_text.split()
        plain_words = plain_text.split()
        for cipher_word, plain_word in zip(cipher_words, plain_words):
            word_map[cipher_word] = plain_word
            if len(cipher_word) != len(plain_word):
                continue
            for cipher_char, plain_char in zip(cipher_word, plain_word):
                if not cipher_char.isalpha() or not plain_char.isalpha():
                    continue
                previous = char_map.get(cipher_char)
                if previous is not None and previous != plain_char:
                    conflicts.add(cipher_char)
                char_map[cipher_char] = plain_char
    if conflicts:
        return None

    query_match = re.search(r"decrypt the following text:\s*([^\n.]+)", prompt_text, flags=re.I)
    if not query_match:
        return None
    cipher_query = query_match.group(1).strip()
    target_words = cipher_query.split()
    gold_words = answer_text.split()
    mappings = []
    for index, cipher_word in enumerate(target_words):
        if cipher_word in word_map:
            decoded = word_map[cipher_word]
        else:
            decoded_chars = [char_map.get(char, "?" if char.isalpha() else char) for char in cipher_word]
            decoded = "".join(decoded_chars)
            if "?" in decoded and index < len(gold_words):
                decoded = gold_words[index]
        if decoded:
            mappings.append(f"{cipher_word}->{decoded}")

    if not mappings:
        return None
    return "Trace: " + "; ".join(mappings) + "."


def build_short_trace(prompt: str, answer_text: str) -> str | None:
    if TRACE_TARGET_MODE == "cipher_short_trace_else_boxed":
        return build_cipher_short_trace(prompt, answer_text)
    raise ValueError(f"Unsupported TRACE_TARGET_MODE: {TRACE_TARGET_MODE!r}")


def format_training_answer(answer: str, prompt: str | None = None) -> str:
    """Return the assistant target text for SFT examples.

    Use raw to reproduce the accepted 0.62 Colab baseline. Use boxed for
    metric-aligned final-answer training. Use short_trace_boxed to teach a
    compact procedural trace for parseable rows, followed by exactly one boxed
    final answer for Kaggle extraction.
    """
    answer_text = "" if answer is None else str(answer).strip()
    boxed = f"\\boxed{{{normalize_answer(answer_text)}}}"
    if TRAIN_TARGET_FORMAT == "raw":
        return answer_text
    if TRAIN_TARGET_FORMAT == "boxed":
        return boxed
    if TRAIN_TARGET_FORMAT == "short_trace_boxed":
        trace = build_short_trace(prompt or "", answer_text)
        return f"{trace}\nFinal: {boxed}" if trace else boxed
    raise ValueError(f"Unsupported TRAIN_TARGET_FORMAT: {TRAIN_TARGET_FORMAT!r}")


def build_sft_text(prompt: str, answer: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n{format_training_answer(answer, prompt)}"


def build_prompt(prompt: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n"


train = train.copy()
train["family"] = train["prompt"].map(infer_family)

sft = train[["id", "family", "prompt", "answer"]].copy()
sft["target_text"] = [format_training_answer(a, p) for p, a in zip(sft["prompt"], sft["answer"])]
sft["text"] = [build_sft_text(p, a) for p, a in zip(sft["prompt"], sft["answer"])]
if EVAL_ROWS and EVAL_ROWS > 0:
    valid = sft.sample(n=min(EVAL_ROWS, len(sft) - 1), random_state=42)
    train_sft = sft.drop(index=valid.index)
else:
    valid = sft.iloc[0:0].copy()
    train_sft = sft.copy()
if TRAIN_ROWS is not None:
    train_sft = train_sft.head(TRAIN_ROWS)
probe_questions = sft.groupby("family", group_keys=False).head(1).head(PROBE_ROWS).reset_index(drop=True)
PROBE_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
probe_questions.to_csv(PROBE_SET_PATH, index=False)
TRACE_SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
sft[["id", "answer", "target_text"]].head(80).to_csv(TRACE_SAMPLE_PATH, index=False)

train_ds = Dataset.from_pandas(train_sft[["text"]], preserve_index=False)
valid_ds = Dataset.from_pandas(valid[["text"]], preserve_index=False) if len(valid) else None
run_config = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "prompt_experiment": PROMPT_EXPERIMENT,
    "system_prompt": SYSTEM_PROMPT,
    "train_target_format": TRAIN_TARGET_FORMAT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "train_rows_config": TRAIN_ROWS,
    "train_rows_actual": len(train_sft),
    "eval_rows": len(valid),
    "probe_rows": len(probe_questions),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "precision_mode": PRECISION_MODE,
    "trace_target_mode": TRACE_TARGET_MODE,
    "trace_sample_path": str(TRACE_SAMPLE_PATH),
    "drive_project_root": str(DRIVE_PROJECT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "run_generated_eval": RUN_GENERATED_EVAL,
    "generated_eval_rows_on_save": GENERATED_EVAL_ROWS_ON_SAVE,
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "resume_checkpoint_step": RESUME_CHECKPOINT_STEP,
    "checkpoint_eval_summary_path": str(CHECKPOINT_EVAL_SUMMARY_PATH),
}
RUN_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
RUN_CONFIG_PATH.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
print(train["family"].value_counts())
print("train rows:", len(train_ds), "valid rows:", 0 if valid_ds is None else len(valid_ds))
print("probe rows:", len(probe_questions), "wrote:", PROBE_SET_PATH)
print("wrote run config:", RUN_CONFIG_PATH)
print("wrote trace samples:", TRACE_SAMPLE_PATH)


## 5. Load model

In [ ]:
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if PRECISION_MODE not in {"bf16", "fp32"}:
    raise ValueError("PRECISION_MODE must be 'bf16' or 'fp32'")
if PRECISION_MODE == "bf16" and not bf16_supported:
    raise RuntimeError("This runtime does not support BF16. Use an A100/H100 or set PRECISION_MODE='fp32'.")

compute_dtype = torch.bfloat16 if PRECISION_MODE == "bf16" else torch.float32
model_dtype = compute_dtype
trainer_bf16 = PRECISION_MODE == "bf16"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("model:", MODEL_NAME)
print("precision mode:", PRECISION_MODE)
print("model dtype:", model_dtype)
print("compute dtype:", compute_dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=model_dtype,
    device_map={"": 0},
    trust_remote_code=True,
    token=HF_TOKEN,
)
model.config.use_cache = False
model.generation_config.use_cache = False
model.eval()


# Known Nemotron BF16 issue: expert-weighted outputs can become FP32 before index_add_.
def _patched_nemotron_moe(self, hidden_states, topk_indices, topk_weights):
    final_hidden_states = torch.zeros_like(hidden_states, dtype=topk_weights.dtype)
    expert_mask = torch.nn.functional.one_hot(topk_indices, num_classes=len(self.experts))
    expert_mask = expert_mask.permute(2, 0, 1)

    for expert_idx in range(len(self.experts)):
        expert = self.experts[expert_idx]
        mask = expert_mask[expert_idx]
        token_indices, weight_indices = torch.where(mask)

        if token_indices.numel() > 0:
            expert_weights = topk_weights[token_indices, weight_indices]
            expert_input = hidden_states[token_indices]
            expert_output = expert(expert_input)
            weighted_output = expert_output * expert_weights.unsqueeze(-1)
            final_hidden_states.index_add_(0, token_indices, weighted_output.to(final_hidden_states.dtype))
        else:
            expert_dtype = expert.down_proj.weight.dtype
            dummy_input = torch.zeros_like(hidden_states[0]).unsqueeze(0).to(expert_dtype)
            final_hidden_states = final_hidden_states + expert(dummy_input).to(final_hidden_states.dtype)

    return final_hidden_states.to(hidden_states.dtype)


def patch_nemotron_moe_dtype(current_model):
    patched = 0
    for module in current_model.modules():
        if module.__class__.__name__ == "NemotronHMOE":
            module.moe = types.MethodType(_patched_nemotron_moe, module)
            patched += 1
    print("patched Nemotron MoE dtype modules:", patched)


if PRECISION_MODE == "bf16":
    patch_nemotron_moe_dtype(model)


## 6. Module summary

In [ ]:
def dtype_name(dtype):
    return str(dtype).replace("torch.", "") if dtype is not None else "-"


module_rows = []
for name, module in model.named_modules():
    class_name = module.__class__.__name__
    if "Linear" in class_name or "Conv" in class_name:
        short_name = name.split(".")[-1]
        weight = getattr(module, "weight", None)
        compute_dtype = getattr(module, "compute_dtype", None)
        param_dtypes = sorted({dtype_name(p.dtype) for p in module.parameters(recurse=False)})
        buffer_dtypes = sorted({dtype_name(b.dtype) for b in module.buffers(recurse=False)})
        module_rows.append({
            "module": short_name,
            "class": class_name,
            "weight_dtype": dtype_name(getattr(weight, "dtype", None)),
            "compute_dtype": dtype_name(compute_dtype),
            "param_dtypes": ", ".join(param_dtypes) or "-",
            "buffer_dtypes": ", ".join(buffer_dtypes) or "-",
        })

module_summary = (
    pd.DataFrame(module_rows)
    .groupby(["module", "class", "weight_dtype", "compute_dtype", "param_dtypes", "buffer_dtypes"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(module_summary)
print("LoRA targets:", LORA_TARGET_MODULES)


## 7. Baseline inference

In [ ]:
def generate_answers(current_model, rows):
    records = []
    was_training = current_model.training
    current_model.eval()
    device = next(current_model.parameters()).device
    for row in rows.itertuples(index=False):
        inputs = tokenizer(build_prompt(row.prompt), return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(device)
        start = time.time()
        with torch.no_grad():
            output_ids = current_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        seconds = time.time() - start
        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        generated_token_count = int(generated_ids.numel())
        hit_max_new_tokens = generated_token_count >= MAX_NEW_TOKENS
        raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        record = {
            "id": row.id,
            "answer": extract_final_answer(raw),
            "seconds": seconds,
            "generated_tokens": generated_token_count,
            "hit_max_new_tokens": hit_max_new_tokens,
            "raw_output": raw,
        }
        if hasattr(row, "family"):
            record["family"] = row.family
        records.append(record)
    if was_training:
        current_model.train()
    return pd.DataFrame(records)


def summarize_generation_snapshot(scored: pd.DataFrame) -> pd.DataFrame:
    def summarize_part(part: pd.DataFrame, family: str) -> dict:
        return {
            "phase": str(part["phase"].iloc[0]) if len(part) else "unknown",
            "step": int(part["step"].iloc[0]) if len(part) else -1,
            "family": family,
            "rows": int(len(part)),
            "exact_matches": int(part["match"].sum()) if len(part) else 0,
            "exact_accuracy": float(part["match"].mean()) if len(part) else 0.0,
            "empty_answers": int(part["empty_answer"].sum()) if "empty_answer" in part else None,
            "hit_max_new_tokens": int(part["hit_max_new_tokens"].sum()) if "hit_max_new_tokens" in part else None,
            "avg_generated_tokens": float(part["generated_tokens"].mean()) if "generated_tokens" in part else None,
            "avg_seconds": float(part["seconds"].mean()) if "seconds" in part else None,
        }

    rows = [summarize_part(scored, "all")]
    if "family" in scored.columns:
        for family, part in scored.groupby("family", dropna=False):
            rows.append(summarize_part(part, str(family)))
    return pd.DataFrame(rows).sort_values(["step", "phase", "family"]).reset_index(drop=True)


def record_generation_snapshot(
    current_model,
    rows,
    step: int,
    phase: str,
    predictions_path: Path,
    summary_path: Path | None = None,
    max_rows: int | None = None,
    append_predictions: bool = True,
    display_predictions: bool = False,
):
    if rows is None or len(rows) == 0:
        print(f"no rows for generation snapshot phase={phase} step={step}")
        return pd.DataFrame(), pd.DataFrame()


    rows = rows.reset_index(drop=True)
    if max_rows is not None:
        rows = rows.head(int(max_rows)).reset_index(drop=True)
    scored = generate_answers(current_model, rows).copy()
    scored.insert(0, "phase", phase)
    scored.insert(1, "step", int(step))
    scored["gold"] = rows["answer"].values
    scored["match"] = scored["gold"].map(normalize_answer) == scored["answer"].map(normalize_answer)
    scored["empty_answer"] = scored["answer"].astype(str).str.len().eq(0)

    predictions_path.parent.mkdir(parents=True, exist_ok=True)
    if append_predictions and predictions_path.exists():
        existing = pd.read_csv(predictions_path)
        if {"phase", "step"}.issubset(existing.columns):
            existing_steps = pd.to_numeric(existing["step"], errors="coerce").fillna(-1).astype(int)
            keep = ~((existing["phase"].astype(str) == phase) & (existing_steps == int(step)))
            predictions = pd.concat([existing[keep], scored], ignore_index=True)
        else:
            predictions = scored
    else:
        predictions = scored
    predictions.to_csv(predictions_path, index=False)


    summary = summarize_generation_snapshot(scored)
    if summary_path is not None:
        summary_path.parent.mkdir(parents=True, exist_ok=True)
        if summary_path.exists():
            existing_summary = pd.read_csv(summary_path)
            if {"phase", "step"}.issubset(existing_summary.columns):
                existing_steps = pd.to_numeric(existing_summary["step"], errors="coerce").fillna(-1).astype(int)
                keep = ~((existing_summary["phase"].astype(str) == phase) & (existing_steps == int(step)))
                summary_to_write = pd.concat([existing_summary[keep], summary], ignore_index=True)
            else:
                summary_to_write = summary
        else:
            summary_to_write = summary
        summary_to_write.to_csv(summary_path, index=False)

    print(f"generation snapshot phase={phase} step={step}; wrote {predictions_path}")
    display(summary)
    if display_predictions:
        display_cols = [col for col in ["phase", "step", "id", "family", "gold", "answer", "match", "generated_tokens", "hit_max_new_tokens", "raw_output"] if col in scored.columns]
        display(scored[display_cols])
    return scored, summary


pd.set_option("display.max_colwidth", 240)
baseline_results, baseline_summary = record_generation_snapshot(
    model,
    probe_questions,
    step=0,
    phase="probe_before",
    predictions_path=PROBE_EVOLUTION_CSV_PATH,
    max_rows=PROBE_ROWS,
    append_predictions=True,
    display_predictions=True,
)


## 8. Train LoRA

In [ ]:
import gc
import math
gc.collect()
torch.cuda.empty_cache()

steps_per_epoch = max(1, math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM_STEPS)))
total_train_steps = max(1, math.ceil(steps_per_epoch * float(EPOCHS)))
logging_steps = max(1, total_train_steps // LOG_EVAL_POINTS)
eval_steps = logging_steps
save_steps = max(logging_steps, total_train_steps // SAVE_POINTS)
eval_strategy = "steps" if valid_ds is not None else "no"
print(
    "train steps:", total_train_steps,
    "logging_steps:", logging_steps,
    "eval_steps:", eval_steps if eval_strategy != "no" else None,
    "save_steps:", save_steps,
)


In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=logging_steps,
    logging_first_step=True,
    save_steps=save_steps,
    eval_strategy=eval_strategy,
    eval_steps=eval_steps if eval_strategy != "no" else None,
    report_to="tensorboard",
    fp16=False,
    bf16=trainer_bf16,
)


class GenerationCallback(TrainerCallback):
    def on_log(self, args, state, control, model=None, **kwargs):
        step = int(state.global_step)
        logs = kwargs.get("logs") or {}
        if model is not None and step > 0 and "loss" in logs:
            record_generation_snapshot(
                model,
                probe_questions,
                step=step,
                phase="probe_log",
                predictions_path=PROBE_EVOLUTION_CSV_PATH,
                max_rows=PROBE_ROWS,
                append_predictions=True,
                display_predictions=True,
            )
        return control

    def on_save(self, args, state, control, model=None, **kwargs):
        if GENERATED_EVAL_ROWS_ON_SAVE and model is not None:
            step = int(state.global_step)
            record_generation_snapshot(
                model,
                valid,
                step=step,
                phase="checkpoint",
                predictions_path=CHECKPOINT_EVAL_DIR / f"generated_eval_predictions_checkpoint-{step}.csv",
                summary_path=CHECKPOINT_EVAL_SUMMARY_PATH,
                max_rows=GENERATED_EVAL_ROWS_ON_SAVE,
                append_predictions=False,
                display_predictions=False,
            )
        return control


trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)
trainer.add_callback(GenerationCallback())


def latest_checkpoint(output_dir: Path):
    checkpoints = []
    for path in output_dir.glob("checkpoint-*"):
        try:
            step = int(path.name.rsplit("-", 1)[-1])
        except ValueError:
            continue
        checkpoints.append((step, path))
    return max(checkpoints, key=lambda item: item[0])[1] if checkpoints else None


existing_checkpoint = latest_checkpoint(OUTPUT_DIR)
if RESUME_FROM_CHECKPOINT:
    if RESUME_CHECKPOINT_STEP is None:
        raise ValueError("Set RESUME_CHECKPOINT_STEP to the checkpoint number, for example 192")
    resume_checkpoint = OUTPUT_DIR / f"checkpoint-{int(RESUME_CHECKPOINT_STEP)}"
    if not resume_checkpoint.is_dir():
        raise FileNotFoundError(f"resume checkpoint not found: {resume_checkpoint}")
    print("resuming from checkpoint:", resume_checkpoint)
    trainer.train(resume_from_checkpoint=str(resume_checkpoint))
elif existing_checkpoint is not None:
    raise RuntimeError(
        f"Existing checkpoint found: {existing_checkpoint}. "
        "Set RESUME_FROM_CHECKPOINT=True and RESUME_CHECKPOINT_STEP to continue, "
        "or use a new EXPERIMENT_NAME / clear OUTPUT_DIR for a clean run."
    )
else:
    print("starting fresh training run")
    trainer.train()

pd.DataFrame(trainer.state.log_history).to_csv(TRAINER_LOG_CSV_PATH, index=False)
print("wrote trainer log:", TRAINER_LOG_CSV_PATH)
trainer.save_model(str(ADAPTER_DIR))


## 9. Final probe and generated eval


In [ ]:
final_step = int(trainer.state.global_step)
record_generation_snapshot(
    trainer.model,
    probe_questions,
    step=final_step,
    phase="probe_final",
    predictions_path=PROBE_EVOLUTION_CSV_PATH,
    max_rows=PROBE_ROWS,
    append_predictions=True,
    display_predictions=True,
)
if RUN_GENERATED_EVAL:
    final_eval, final_summary = record_generation_snapshot(
        trainer.model,
        valid,
        step=final_step,
        phase="final",
        predictions_path=GENERATED_EVAL_CSV_PATH,
        summary_path=GENERATED_EVAL_SUMMARY_PATH,
        append_predictions=False,
        display_predictions=True,
    )
else:
    print("RUN_GENERATED_EVAL=False; skipping final generated-answer eval.")


## 10. Training decision dashboard


In [ ]:
trainer_log = pd.DataFrame(trainer.state.log_history)
trainer_log.to_csv(TRAINER_LOG_CSV_PATH, index=False)
print("wrote trainer log:", TRAINER_LOG_CSV_PATH)

display_cols = [col for col in ["step", "epoch", "loss", "eval_loss", "learning_rate", "grad_norm"] if col in trainer_log.columns]
display(trainer_log[display_cols].tail(30))

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 4))
    if {"step", "loss"}.issubset(trainer_log.columns):
        train_loss = trainer_log.dropna(subset=["loss"])
        if len(train_loss):
            ax.plot(train_loss["step"], train_loss["loss"], marker="o", label="train loss")
    if {"step", "eval_loss"}.issubset(trainer_log.columns):
        eval_loss_df = trainer_log.dropna(subset=["eval_loss"])
        if len(eval_loss_df):
            ax.plot(eval_loss_df["step"], eval_loss_df["eval_loss"], marker="o", label="eval loss")
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("loss")
    ax.set_title(EXPERIMENT_NAME)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()
except Exception as exc:
    print("loss plot skipped:", exc)

probe_evolution = pd.read_csv(PROBE_EVOLUTION_CSV_PATH) if PROBE_EVOLUTION_CSV_PATH.exists() else pd.DataFrame()
if len(probe_evolution):
    probe_evolution["match_bool"] = probe_evolution["match"].astype(str).str.lower().isin(["true", "1"])
    probe_summary = (
        probe_evolution.groupby(["phase", "step"], as_index=False)
        .agg(
            probe_matches=("match_bool", "sum"),
            probe_rows=("match_bool", "count"),
            avg_seconds=("seconds", "mean"),
        )
        .sort_values("step")
    )
    probe_summary["probe_match_rate"] = probe_summary["probe_matches"] / probe_summary["probe_rows"]
    display(probe_summary)

    compact_probe = probe_evolution[["phase", "step", "id", "gold", "answer", "match", "raw_output"]].copy()
    display(compact_probe.tail(PROBE_ROWS * 3))
else:
    probe_summary = pd.DataFrame()
    print("No probe evolution file found yet:", PROBE_EVOLUTION_CSV_PATH)

latest_train_loss = None
if "loss" in trainer_log.columns and trainer_log["loss"].notna().any():
    latest_train_loss = float(trainer_log.dropna(subset=["loss"])["loss"].iloc[-1])

best_eval_loss = None
latest_eval_loss = None
if "eval_loss" in trainer_log.columns and trainer_log["eval_loss"].notna().any():
    eval_losses = trainer_log.dropna(subset=["eval_loss"])
    best_eval_loss = float(eval_losses["eval_loss"].min())
    latest_eval_loss = float(eval_losses["eval_loss"].iloc[-1])

final_probe_matches = None
final_probe_rows = None
final_probe_phase = None
if len(probe_evolution):
    final_step = probe_evolution["step"].max()
    final_probe_candidates = probe_evolution[probe_evolution["step"] == final_step].copy()
    phase_priority = {"probe_final": 0, "probe_log": 1, "probe_before": 2}
    final_probe_candidates["phase_priority"] = final_probe_candidates["phase"].map(phase_priority).fillna(9)
    final_probe_phase = final_probe_candidates.sort_values("phase_priority")["phase"].iloc[0]
    final_probe = final_probe_candidates[final_probe_candidates["phase"] == final_probe_phase].copy()
    final_probe["match_bool"] = final_probe["match"].astype(str).str.lower().isin(["true", "1"])
    final_probe_matches = int(final_probe["match_bool"].sum())
    final_probe_rows = int(len(final_probe))

submit_signals = pd.DataFrame([
    {"signal": "latest_train_loss", "value": latest_train_loss, "how_to_use": "Check for obvious failed training or divergence."},
    {"signal": "latest_eval_loss", "value": latest_eval_loss, "how_to_use": "Prefer lower than prior local experiments, but do not overtrust it."},
    {"signal": "best_eval_loss", "value": best_eval_loss, "how_to_use": "Useful if latest eval worsens late in training."},
    {"signal": "final_probe_matches", "value": None if final_probe_matches is None else f"{final_probe_matches}/{final_probe_rows} ({final_probe_phase})", "how_to_use": "Small sample only; inspect raw outputs for formatting failures."},
])
display(submit_signals)


## 11. Sanity predictions


In [ ]:
sanity_predictions = generate_answers(trainer.model, test)
sanity_submission = sanity_predictions[["id", "answer"]].copy()

assert list(sanity_submission.columns) == ["id", "answer"]
assert sanity_submission["id"].tolist() == test["id"].tolist()
assert sanity_submission["answer"].astype(str).str.len().gt(0).all()

SANITY_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
sanity_submission.to_csv(SANITY_CSV_PATH, index=False)
sanity_predictions.to_csv(SANITY_RAW_CSV_PATH, index=False)
print("wrote sanity CSV:", SANITY_CSV_PATH)
print("wrote raw sanity CSV:", SANITY_RAW_CSV_PATH)
display(sanity_predictions[["id", "answer", "seconds", "generated_tokens", "hit_max_new_tokens", "raw_output"]])


## 12. Package run bundle


In [ ]:
def validate_adapter_dir(adapter_dir: Path) -> list[Path]:
    required = ["adapter_config.json", "adapter_model.safetensors"]
    missing = [name for name in required if not (adapter_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"missing required adapter files in {adapter_dir}: {missing}")
    return [adapter_dir / name for name in required]


def add_file_once(archive: zipfile.ZipFile, path: Path, seen: set[str], arcname: str) -> None:
    if not path.exists():
        print("missing bundle file:", path)
        return
    if not path.is_file():
        return
    arcname = arcname.replace("\\", "/")
    if arcname in seen:
        return
    archive.write(path, arcname=arcname)
    seen.add(arcname)


def write_run_bundle_zip(output_zip: Path) -> None:
    """Write one portable run archive for local packaging and reports.

    Kaggle submission.zip should be built locally from bundle adapter files,
    not uploaded from this Colab artifact directly.
    """
    adapter_files = validate_adapter_dir(ADAPTER_DIR)
    bundle_files = [
        RUN_CONFIG_PATH,
        PROBE_SET_PATH,
        PROBE_EVOLUTION_CSV_PATH,
        TRAINER_LOG_CSV_PATH,
        TRACE_SAMPLE_PATH,
        SANITY_CSV_PATH,
        SANITY_RAW_CSV_PATH,
        GENERATED_EVAL_CSV_PATH,
        GENERATED_EVAL_SUMMARY_PATH,
        OUTPUT_DIR / "README.md",
    ]
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    if output_zip.exists():
        output_zip.unlink()

    seen = set()
    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in adapter_files:
            add_file_once(archive, path, seen, f"adapter/{path.name}")
        for path in bundle_files:
            add_file_once(archive, path, seen, f"diagnostics/{path.name}")
        for path in sorted(OUTPUT_DIR.rglob("events.out.tfevents*")):
            add_file_once(archive, path, seen, f"tensorboard/{path.name}")

        if CHECKPOINT_EVAL_DIR.exists():
            for path in sorted(CHECKPOINT_EVAL_DIR.rglob("*")):
                if path.is_file():
                    rel = path.relative_to(CHECKPOINT_EVAL_DIR).as_posix()
                    add_file_once(archive, path, seen, f"checkpoint_eval/{rel}")

    with zipfile.ZipFile(output_zip) as archive:
        names = sorted(archive.namelist())
    for name in ["adapter/adapter_config.json", "adapter/adapter_model.safetensors"]:
        assert name in names, names
    print("wrote run bundle:", output_zip)
    display(pd.DataFrame({"bundle_file": names}))


write_run_bundle_zip(RUN_BUNDLE_ZIP_PATH)


## 13. Download


In [ ]:
files.download(str(RUN_BUNDLE_ZIP_PATH))
